## Parsing: Docling anchor + VLM

Exploratory test variant of the Docling anchored parser: importing the stack, reporting the GPU and picking the SVM lecture as the single test document

In [ ]:
import time
from openai import OpenAI
from dotenv import load_dotenv
import json
import re
import os
import base64
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from IPython.display import display, HTML
from io import BytesIO

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

print("PyTorch:", torch.__version__)
print("CUDA verfuegbar:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

PDF_PATH = Path(os.getcwd()).parent / "data" / "raw" / "pdfs"

MODUL = "machine_learning"
TEST_PRESENTATION = next(PDF_PATH.glob("*svm*.pdf"))
TEST_PRESENTATION.name

## Docling: parse the document and produce per page anchor text

Running Docling to produce per page anchor Markdown and page images, the structural skeleton fed to the VLM

In [ ]:
options = PdfPipelineOptions()
options.do_ocr = False
options.do_table_structure = True
options.do_formula_enrichment = True
options.generate_page_images = True
options.images_scale = 2

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=options)}
)

print("Docling startet Parsing ...")
start = time.time()
result = converter.convert(TEST_PRESENTATION)
document = result.document
print(f"Fertig in {time.time() - start:.1f} Sekunden.")
print("Seiten:", len(document.pages))

pages: dict[int, Image.Image] = {}
anker_texts: dict[int, str] = {}
for page in result.pages:
    pages[page.page_no] = page.image
    anker_texts[page.page_no] = document.export_to_markdown(page_no=page.page_no)

print(f"Seiten-Indizes: {sorted(pages.keys())}")

## Define the chunk schema

Defining the slide schema and loading the anchor aware system prompt that constrains the VLM output

In [ ]:
import json
from pydantic import BaseModel, Field

from system_prompt import get_system_prompt_with_anchor

class VlmSlideOutput(BaseModel):
    title: str = Field(
        default="",
        description="Titel der Folie. Leer-String wenn kein Titel sichtbar."
    )

    page_content: str = Field(
        default="",
        description=(
            "Vollständige strukturierte Markdown-Beschreibung der gesamten Folie: "
            "alle sichtbaren Texte (verbatim), Aufzählungs-Hierarchie, Diagramme, "
            "[Formeln], [Code], [Grafik]. Alle sichtbaren Informationen erfassen, nichts weglassen. "
        )
    )

class SlideChunk(BaseModel):
    id: str
    page_numbers: list[int]
    page_reference_path: str
    modul: str
    lecture: str
    title: str
    page_content: str

SCHEMA_JSON = json.dumps(VlmSlideOutput.model_json_schema(), indent=2, ensure_ascii=False)

SYSTEM_PROMPT_TEMPLATE = get_system_prompt_with_anchor()

Configuring the VLM client and parse_slide_by_vlm, which combines the page image with the Docling anchor text and validates the JSON

In [ ]:
load_dotenv("../.env", override=True)

client = OpenAI(
    base_url=os.getenv("GATEWAY_URL", ""),
    api_key=os.getenv("BEARER_TOKEN", ""),
)
model = os.getenv("VL_MODEL_GATEWAY", "")
assert model, "VL_MODEL_GATEWAY env-var nicht gesetzt"
print(f"Model: {model}")

MAX_RETRIES = 5

def encode_image(image_path: Path) -> str:
    return base64.b64encode(image_path.read_bytes()).decode("utf-8")

def parse_slide_by_vlm(image_path: Path, anker_text: str) -> VlmSlideOutput:
    b64 = encode_image(image_path)
    system_prompt = SYSTEM_PROMPT_TEMPLATE + "\n" + (anker_text or "")

    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": "Beschreibe die folgende Vorlesungsfolie wie im Systemprompt gefordert"},
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/png;base64,{b64}"
                                },
                            },
                        ],
                    },
                ],
                response_format={"type": "json_object"},
                extra_body={
                    "chat_template_kwargs": {
                        "enable_thinking": False,
                    }
                },
                max_tokens=8192,
                temperature=0,
            )

            choice = response.choices[0]
            content = choice.message.content
            reasoning = getattr(choice.message, "reasoning_content", None)

            if not content:
                print(f"Empty: finish_reason={choice.finish_reason}, "
                      f"reasoning_len={len(reasoning or '')}, "
                      f"usage={response.usage}")
                raise ValueError("empty content")

            return VlmSlideOutput.model_validate_json(content)

        except Exception as e:
            print(
                f"[Retry {attempt + 1}/{MAX_RETRIES}] "
                f"Error parsing: {image_path.name}: {e}"
            )

            if attempt == MAX_RETRIES - 1:
                raise

            time.sleep(1)

## Turn the PDF into slides and transcribe them via Docling anchor + VLM

Parsing every page with the anchored VLM and writing the _docling_anker.json chunks for this lecture

In [ ]:
import json

lecture = TEST_PRESENTATION.stem

out_reference = Path("../data/reference_slides") / MODUL / lecture
out_chunking_json = Path("../data/parsed") / MODUL / lecture / f"{lecture}_chunks_docling_anker.json"

out_reference.mkdir(parents=True, exist_ok=True)
out_chunking_json.parent.mkdir(parents=True, exist_ok=True)

chunks: list[SlideChunk] = []

page_nos = sorted(pages.keys())
total = len(page_nos)

for i, page_no in enumerate(page_nos, start=1):
    img_path = out_reference / f"page_{page_no}.png"

    if not img_path.exists():
        img = pages[page_no]
        if img is None:
            raise RuntimeError(f"No slide image: {page_no} (generate_page_images=True?)")
        img.convert("RGB").save(img_path)

    anker = anker_texts.get(page_no, "")
    print(f"[{i:>3}/{total}] {img_path.name} -> VLM (+Anker) ...", end=" ", flush=True)
    vlm_data = parse_slide_by_vlm(img_path, anker)
    print("ok")

    chunk_fields = vlm_data.model_dump()
    chunks.append(SlideChunk(
        id=f"{lecture}_page_{page_no}",
        lecture=lecture,
        page_numbers=[page_no],
        page_reference_path=str(img_path),
        modul=MODUL,
        **chunk_fields,
    ))

out_chunking_json.write_text(
    json.dumps([c.model_dump() for c in chunks], indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"\n{len(chunks)} chunks gespeichert: {out_chunking_json}")

## Display the results

Displaying each slide next to its parsed content for a visual inspection of the result

In [ ]:
from IPython.display import display, HTML
import base64, html as html_lib, json, pathlib

data = json.loads(out_chunking_json.read_text(encoding="utf-8"))

for c in data:                     
  img_path = c.get('page_reference_path')
  img_data = ""
  if img_path and pathlib.Path(img_path).is_file():
      b = pathlib.Path(img_path).read_bytes()
      img_data = "data:image/png;base64," + base64.b64encode(b).decode('ascii')

  page_content = c.get("page_content", "")
  pretty_content = html_lib.escape(page_content).replace("\n", "<br>")
  pretty_json = html_lib.escape(json.dumps(c, ensure_ascii=False, indent=2)).replace("\n", "<br>")

  html = f"""
  <div style="display:flex; gap:16px; align-items:flex-start;">
    <div style="flex:1 1 50%; border:1px solid #ddd; padding:8px; background:#fff;">
      <img src="{img_data}" style="max-width:100%; height:auto; display:block;">
    </div>
    <div style="flex:1 1 50%; border:1px solid #ddd; padding:12px; background:#fafafa; overflow:auto; max-height:80vh;">
      <div style="font-family:system-ui, sans-serif; font-size:14px; line-height:1.5; color:#222;">
        <div style="font-size:18px; font-weight:700; margin-bottom:10px;">{html_lib.escape(str(c.get('title', '')))}</div>
        <div style="margin-bottom:10px; color:#555;"><strong>Seiten:</strong> {html_lib.escape(str(c.get('page_numbers', [])))}</div>
        <div style="margin-bottom:14px;"><strong>Page content</strong></div>
        <div style="white-space:pre-wrap; font-family:inherit; font-size:13px; line-height:1.6; background:#fff; border:1px solid #e3e3e3; padding:12px; border-radius:6px;">{pretty_content}</div>
        <details style="margin-top:14px;">
          <summary style="cursor:pointer; font-weight:600;">Rohdaten anzeigen</summary>
          <pre style="white-space:pre-wrap; font-family:monospace; font-size:12px; color:#000; margin-top:10px; background:#fff; border:1px solid #e3e3e3; padding:12px; border-radius:6px;">{pretty_json}</pre>
        </details>
      </div>
    </div>
  </div>
  """

  display(HTML(html))